# Distribution and scatter plots of all factors

This notebook adds two figures to the descriptive analysis of notebook 6:

1. A **boxplot** of charging point density, which shows the distribution and makes outliers visible.
2. **Scatter plots** of charging point density against each explanatory variable, with a least-squares regression line.

The choice of plot follows the data relationship we want to show: a boxplot for a **distribution** with outliers, and scatter plots for the **correlation** between two numerical variables.

**Input:** `data/processed/all_variables_per_municipality.csv`, created by notebook 6. Run notebook 6 first.

**Outputs:** `figures/figure_boxplot_charging_point_density.png`, `figures/figure_scatter_all_factors.png`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

PROCESSED = "../data/processed/"
FIGURES = "../figures/"

## 1. Load the data

The same readable English labels are used as in notebook 6.

In [ ]:
df = pd.read_csv(PROCESSED + "all_variables_per_municipality.csv")

LABELS = {
    "laadpunten_per_1000_huishoudens": "Charging points per 1,000 households",
    "gem_gestandaardiseerd_inkomen_keur": "Mean standardised income (k€)",
    "med_gestandaardiseerd_inkomen_keur": "Median standardised income (k€)",
    "jongeren_onder_25_pct": "Residents under 25 (%)",
    "ouderen_65plus_pct": "Residents 65+ (%)",
    "linkse_stemmen_pct": "Left-wing vote share (%)",
    "linkse_stemmen_smal_pct": "Left-wing vote share, narrow (%)",
    "hoogopgeleid_pct": "Highly educated (%)",
    "bevolkingsdichtheid": "Population density (inh./km²)",
    "meergezinswoning_pct": "Apartments (% of dwellings)",
}
data = df[list(LABELS)].rename(columns=LABELS)
target = "Charging points per 1,000 households"

print("Municipalities:", len(data))

## 2. Boxplot of charging point density

The box runs from the first quartile (Q1) to the third quartile (Q3); the red line is the median. Values more than 1.5 × the interquartile range (IQR = Q3 − Q1) above Q3 or below Q1 are shown as separate points: these are the outliers. Each outlier is labelled with its municipality name.

In [ ]:
y = data[target]
names = df["gemeente"]

# Quartiles and fences for outliers
q1 = y.quantile(0.25)
q3 = y.quantile(0.75)
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
lower_fence = q1 - 1.5 * iqr

outliers = y[(y > upper_fence) | (y < lower_fence)]
print(f"Q1 = {q1:.1f}, median = {y.median():.1f}, Q3 = {q3:.1f}, upper fence = {upper_fence:.1f}")
for i in outliers.index:
    print("Outlier:", names[i], round(y[i], 1))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
ax.boxplot(y, vert=False, widths=0.5, patch_artist=True,
           boxprops={"facecolor": "#c6dbef", "edgecolor": "#1d639b"},
           medianprops={"color": "#d32f2f", "linewidth": 2},
           flierprops={"marker": "o", "markerfacecolor": "#1d639b", "markersize": 5})

# Label every outlier with its municipality name
for i in outliers.index:
    ax.annotate(names[i], (y[i], 1), textcoords="offset points", xytext=(0, 20),
                ha="center", fontsize=8)

ax.set_yticks([])
ax.set_xlabel("Charging points per 1,000 households")
ax.set_title(f"Boxplot of public charging point density (N = {len(y)}); outliers above {upper_fence:.1f}")
plt.tight_layout()
plt.savefig(FIGURES + "figure_boxplot_charging_point_density.png", dpi=200)
plt.show()

## 3. Scatter plots of all factors

Each panel shows charging point density against one explanatory variable. The red line is the least-squares fit and the shaded band its 95% confidence interval; the Pearson correlation r and the number of municipalities N are given in the title.

Population density is strongly right-skewed, so it is plotted on a logarithmic axis and the line is fitted on log(density), as in the regression (notebook 7).

Seaborn calculates the confidence band with random resampling (bootstrapping); `seed=0` fixes the random numbers, so the figure is exactly the same every time the notebook is run.

In [ ]:
explanatory = [col for col in data.columns if col != target]   # 9 variables -> 3 x 3 grid

fig, axes = plt.subplots(3, 3, figsize=(14, 12))
axes = axes.flatten()

for i, col in enumerate(explanatory):
    ax = axes[i]
    pair = data[[col, target]].dropna()

    if col == "Population density (inh./km²)":
        # Skewed variable: log axis, line and r based on log(density)
        sns.regplot(data=pair, x=col, y=target, ax=ax, logx=True, seed=0,
                    scatter_kws={"s": 12, "alpha": 0.5, "color": "#1d639b"},
                    line_kws={"color": "#d32f2f", "linewidth": 2})
        ax.set_xscale("log")
        r = stats.pearsonr(np.log(pair[col]), pair[target])[0]
        ax.set_title(f"{col}\nr = {r:.2f} (log scale), N = {len(pair)}", fontsize=10)
    else:
        sns.regplot(data=pair, x=col, y=target, ax=ax, seed=0,
                    scatter_kws={"s": 12, "alpha": 0.5, "color": "#1d639b"},
                    line_kws={"color": "#d32f2f", "linewidth": 2})
        r = stats.pearsonr(pair[col], pair[target])[0]
        ax.set_title(f"{col}\nr = {r:.2f}, N = {len(pair)}", fontsize=10)

    ax.set_xlabel("")
    if i % 3 == 0:                                   # y-label only in the left column
        ax.set_ylabel("Charging points per 1,000 hh")
    else:
        ax.set_ylabel("")

fig.suptitle("Charging point density against each explanatory variable (red line: least-squares fit)", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES + "figure_scatter_all_factors.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. Interpretation

> ** Still to do**